# 08 - Model Serving on SPCS

This notebook demonstrates deploying a model from the registry as a real-time REST API endpoint on Snowpark Container Services (SPCS).

**Key Concepts**:
- One-line deployment from Model Registry to SPCS
- Auto-generated REST API with OpenAPI spec
- Auto-scaling containers
- Callable via REST, SQL, or Python
- Data never leaves Snowflake's security perimeter

**Prerequisites**:
- Run `02_model_training_xgboost.ipynb` first
- Compute pools must exist (run `sql/08_setup_compute_pool.sql`)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
print(f"Connected: {session.get_current_user()}")

In [ ]:
# Verify compute pools exist
session.sql("SHOW COMPUTE POOLS").show()
print("\nRequired pools: ML_SERVING_POOL (for serving), ML_CPU_POOL (for image build)")

In [ ]:
# Load model from registry
reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)
mv = reg.get_model("CHURN_PREDICTION_MODEL").default
print(f"Loaded: {mv.model_name} version {mv.version_name}")

In [ ]:
# Deploy model as a service on SPCS
# This builds a container image and deploys it to the serving pool
#
# NOTE: This requires active compute pools. If pools are not available,
# review the code to understand the deployment pattern.

try:
    service = mv.create_service(
        service_name="CHURN_PREDICTION_SERVICE",
        service_compute_pool="ML_SERVING_POOL",
        image_build_compute_pool="ML_CPU_POOL",
        ingress_enabled=True
    )
    print(f"Service deployed: CHURN_PREDICTION_SERVICE")
    print(f"Endpoint: {service.get_endpoint_url()}")
except Exception as e:
    print(f"Service deployment note: {e}")
    print("\nThis is expected if compute pools are not active.")
    print("In a live demo, ensure ML_SERVING_POOL and ML_CPU_POOL are ACTIVE.")

## Calling the Service

Once deployed, the model service can be called three ways:

### 1. REST API (external applications)
```bash
curl -X POST https://<endpoint>/predict \
  -H "Authorization: Bearer <token>" \
  -H "Content-Type: application/json" \
  -d '{"data": [[650, 35, 5, 75000, 2, 1, 1, 50000]]}'
```

### 2. SQL (from within Snowflake)
```sql
SELECT 
    CUSTOMER_ID,
    MLOPS_DEMO_DB.PIPELINES.CHURN_PREDICTION_SERVICE!predict(
        CREDIT_SCORE, AGE, TENURE, BALANCE,
        NUM_OF_PRODUCTS, HAS_CR_CARD, IS_ACTIVE_MEMBER, ESTIMATED_SALARY
    ) AS CHURN_PREDICTION
FROM MLOPS_DEMO_DB.RAW.CUSTOMERS
LIMIT 10;
```

### 3. Python (from notebooks or applications)
```python
import requests
response = requests.post(
    f"{endpoint_url}/predict",
    headers={"Authorization": f"Bearer {token}"},
    json={"data": [[650, 35, 5, 75000, 2, 1, 1, 50000]]}
)
print(response.json())
```

In [ ]:
# Check service status (if deployed)
try:
    session.sql("""
    SHOW SERVICES LIKE 'CHURN_PREDICTION_SERVICE' IN SCHEMA MLOPS_DEMO_DB.PIPELINES
    """).show()
except Exception as e:
    print(f"No services found: {e}")

In [ ]:
# Service status and logs (if deployed)
try:
    status = session.sql("""
    SELECT SYSTEM$GET_SERVICE_STATUS('MLOPS_DEMO_DB.PIPELINES.CHURN_PREDICTION_SERVICE')
    """).collect()
    print(f"Service status: {status[0][0]}")
except Exception as e:
    print(f"Service not active: {e}")

In [ ]:
# Cleanup (uncomment to remove the service)
# mv.delete_service("CHURN_PREDICTION_SERVICE")
# print("Service deleted.")

print("Service cleanup: uncomment the lines above when done with the demo.")

## What to Show in SnowSight

Navigate to **AI/ML > Model Registry > CHURN_PREDICTION_MODEL > Deploy** to see:
- Deployed services and their status
- Endpoint URLs
- Container logs

Navigate to **Admin > Compute Pools** to see:
- ML_SERVING_POOL status and utilization
- Container runtime metrics

**Key message**: Going from a registered model to a production REST API is a single Python call. No Docker expertise, no Kubernetes, no API gateway configuration.

---

**Next**: `09_distributed_training.ipynb` - Multi-node distributed training